# Business Entity Resolution — Google Colab runbook

Before running: upload and extract the complete `student_resource` folder to Google Drive (including `dataset/`, `src/`, and `utils/`). The setup cell searches MyDrive and detects its location. This dataset is about 2.3 GB uncompressed. In Colab, choose **Runtime → Change runtime type → T4 GPU**; a high-RAM runtime is recommended. The code also runs on CPU, but blocking will take longer. Keep the project folder on Drive so outputs survive a runtime reset.

Run cells in order. Training defaults to a 150,000 Source-1 sample for model fitting; change `TRAIN_SAMPLE` below if your runtime has enough RAM/time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
DRIVE_ROOT = Path('/content/drive/MyDrive')
matches = [p for p in DRIVE_ROOT.rglob('train_source1.tsv') if p.parent.name == 'train' and p.parent.parent.name == 'dataset']
if not matches:
    print('Could not find dataset/train/train_source1.tsv in MyDrive.')
    print('MyDrive folders:', [p.name for p in DRIVE_ROOT.iterdir()])
    print('ZIP files:', [str(p) for p in DRIVE_ROOT.rglob('*.zip')][:20])
    raise FileNotFoundError('Upload and extract the complete student_resource folder to Drive, then rerun this cell.')
PROJECT = matches[0].parents[2]
assert (PROJECT / 'src/train.py').exists() and (PROJECT / 'src/er_normalize.py').exists(), f'Found dataset at {PROJECT}, but project source files are missing there.'
os.chdir(PROJECT)
print('Project:', PROJECT)
print('GPU available:', subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip())
!df -h /content

In [ ]:
%pip -q install pandas pyarrow numpy scipy scikit-learn lightgbm rapidfuzz anyascii sparse_dot_topn
import pandas, pyarrow, lightgbm, rapidfuzz, sparse_dot_topn
print('Dependencies ready')

## 1. Preprocess data

This writes normalized Parquet files under `work/`. Run both cells once per fresh Colab runtime. The files are on Drive, so they remain available after disconnects.

In [ ]:
!python src/prep.py train

## 1a. Measure candidate-generation recall

This validation step estimates the best score blocking could permit before model errors. If recall is too low, tune candidate generation before spending time on full training. It may take several minutes.

In [ ]:
!python src/eval_block.py

In [ ]:
!python src/prep.py test

## 2. Train and tune the matcher

Training creates `work/lgb.txt` and `work/model_meta.json`, and prints validation scores. This can take a long time and use substantial RAM. If Colab disconnects, rerun preprocessing only if the Parquet files are missing, then rerun training.

In [ ]:
TRAIN_SAMPLE = 150_000
!python src/train.py {TRAIN_SAMPLE}

## 3. Predict the test set and validate submission files

Inference writes both required TSVs into `output/`. The validator's default mode checks formatting without loading all target IDs into memory.

In [ ]:
!python src/run_test.py
!python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test
print('Output files:', (PROJECT / 'output/matching_results.tsv').stat().st_size, 'bytes matching;', (PROJECT / 'output/candidate_pairs.tsv').stat().st_size, 'bytes candidates')

## If a run stops

- `Dataset not found`: check that Drive contains `MyDrive/student_resource/dataset/train/train_source1.tsv`. Update `PROJECT` in the first code cell if your folder is elsewhere.
- CUDA out of memory: switch to a higher-memory GPU runtime or set `ER_BACKEND=cpu` before training/inference (slower, avoids GPU memory use).
- RAM/runtime reset during preprocessing: run the preprocessing cell for the split that did not finish. Each source Parquet is saved when its source finishes.
- Runtime reset during training: preprocessing outputs remain on Drive; rerun the training cell.

The challenge rules prohibit external business lookup. This pipeline uses only the supplied dataset.